# Phase 3: Bias-Variance Trade-off

Empirically verify the decomposition by training polynomial regressors of varying complexity on bootstrap-resampled datasets.

$$\mathbb{E}[(y - \hat{f})^2] = \underbrace{(\mathbb{E}[\hat{f}] - f)^2}_{\text{Bias}^2} + \underbrace{\mathbb{E}[\hat{f}^2] - (\mathbb{E}[\hat{f}])^2}_{\text{Variance}} + \underbrace{\sigma^2}_{\text{Irreducible}}$$

**Ground truth:** $f(x) = x + x^2 - 0.5x^3$ — a cubic polynomial that low-degree models can actually fit, producing the classic U-shaped curve.

In [ ]:
import sys
from pathlib import Path

project_root = str(Path.cwd().parent) if Path.cwd().name == "apps" else str(Path.cwd())
if project_root not in sys.path:
    sys.path.append(project_root)

import matplotlib.pyplot as plt
import numpy as np
from core.prob.bias_variance import (
    PolynomialRegressor,
    bias_variance_decomposition,
    bootstrap_datasets,
    generate_data,
    polynomial_3,
    train_on_bootstraps,
)

### 1. Generate data

Ground truth $f(x) = x + x^2 - 0.5x^3$, with Gaussian noise $\varepsilon \sim \mathcal{N}(0, 0.2^2)$. Using $x \in [-1.5, 1.5]$ so the cubic's curvature is clearly visible — a linear fit will have high bias.

In [ ]:
rng = np.random.default_rng(42)
x, y = generate_data(
    polynomial_3, n_samples=60, noise_std=0.2, x_range=(-1.5, 1.5), rng=rng
)
boots = bootstrap_datasets(x, y, n_bootstraps=200, rng=rng)

x_test = np.linspace(-1.5, 1.5, 200)
f_true = polynomial_3(x_test).flatten()

# Preview the data
fig, ax = plt.subplots(figsize=(8, 3))
xs = np.linspace(-1.5, 1.5, 200)
ax.plot(xs, polynomial_3(xs), "k-", label=r"$f(x) = x + x^2 - 0.5x^3$", linewidth=2)
ax.scatter(x, y, s=15, alpha=0.6, label="Noisy observations")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()

### 2. Sweep model complexity

Fit polynomials of degree 0 through 10 on each bootstrap sample and measure the three components.

In [ ]:
degrees = range(0, 11)
avg_bias_sq = []
avg_variance = []

for d in degrees:
    models = train_on_bootstraps(PolynomialRegressor, boots, {"degree": d})
    result = bias_variance_decomposition(models, x_test, f_true, noise_std=0.2)
    avg_bias_sq.append(result["avg_bias_sq"])
    avg_variance.append(result["avg_variance"])

avg_total = [b + v + 0.2**2 for b, v in zip(avg_bias_sq, avg_variance)]

for d, b, v in zip(degrees, avg_bias_sq, avg_variance):
    print(f"degree={d:2d}:  Bias²={b:.4f}  Var={v:.4f}  Total={b + v + 0.04:.4f}")

### 3. The U-shaped curve

Watch Bias² drop as we add capacity, then Variance take over as we overfit.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(degrees, avg_bias_sq, "o-", label=r"Bias$^2$", color="#E24A33", linewidth=1.5)
ax.plot(degrees, avg_variance, "s-", label="Variance", color="#348ABD", linewidth=1.5)
ax.axhline(0.2**2, color="gray", linestyle="--", label=r"Irreducible $\sigma^2$")
ax.plot(degrees, avg_total, "D-", label="Total Error", color="black", linewidth=2)

# Mark the sweet spot
best_idx = int(np.argmin(avg_total))
ax.scatter(
    [best_idx],
    [avg_total[best_idx]],
    color="green",
    s=120,
    zorder=5,
    label=f"Sweet spot: degree={best_idx}",
)

ax.set_xlabel("Polynomial Degree (model complexity)")
ax.set_ylabel("Error")
ax.set_title(
    r"Bias-Variance Trade-off: $f(x)=x+x^2-0.5x^3$, $x\in[-1.5,1.5]$, $\sigma=0.2$"
)
ax.legend()
ax.set_xticks(list(degrees))
ax.grid(True, alpha=0.3)
fig.tight_layout()

### What to look for

- **Left side (degree 0)**: Bias² dominates heavily — a constant model can't capture the cubic over $[-1.5, 1.5]$
- **Middle (degree 1–5)**: Adding capacity rapidly reduces Bias², Variance stays manageable
- **Right side (degree 6+)**: Variance climbs as high-degree coefficients become unstable
- **$\sigma^2$ line**: The irreducible floor

> Expanding the x range from $[0, 1]$ to $[-1.5, 1.5]$ makes the cubic's curvature much more pronounced, so the drop in Bias² from degree 0 → degree 3 is dramatic, creating a clear U-shaped total error.

In [ ]:
# Appendix: degree=1 (underfit) vs degree=8 (overfit) — 20 fitted curves each
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

for idx, degree in enumerate([1, 8]):
    ax = [ax1, ax2][idx]
    models = train_on_bootstraps(PolynomialRegressor, boots[:20], {"degree": degree})
    ax.plot(xs, polynomial_3(xs), "k-", linewidth=2, label="Truth")
    ax.scatter(x, y, s=10, alpha=0.4, color="gray")
    for m in models:
        ax.plot(xs, m.predict(xs), color="steelblue", alpha=0.3, linewidth=0.5)
    ax.set_title(f"Degree={degree}: 20 fitted curves")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.legend()
    ax.grid(True, alpha=0.3)

fig.tight_layout()